In [1]:
from music21 import note, chord
import os
from collections import defaultdict
from music21 import converter
import pickle

In [ ]:
from music21 import note, chord
import os
from collections import defaultdict
from music21 import converter
import pickle

def get_measure_rhythm(measure):
    rhythm = []
    for el in measure.notesAndRests:
        rhythm.append(el.duration.quarterLength)
    return tuple(rhythm)

def has_spillover(measure):
    for n in measure.notes:
        if n.tie and n.tie.type in ("start", "continue"):
            return True
    return False


CIPI_PATH = "/Volumes/TOSHIBA  2T/CIPI_dataset/scores"


rhythm_dict = defaultdict(lambda: defaultdict(int))

for root, _, files in os.walk(CIPI_PATH):
    for file in files:
        if file.startswith("._"):
            continue
        
        if not file.lower().endswith((".xml", ".musicxml", ".mid", ".midi")):
            continue

        filepath = os.path.join(root, file)

        try:
            score = converter.parse(filepath)
        except Exception as e:
            print(f"Skipping {file}: {e}")
            continue

        # First staff (right hand)
        part = score.parts[0]

    

        # Get time signatures as a list
        ts = part.recurse().getElementsByClass('TimeSignature')
        if not ts: #if empty list
            continue
        meter = ts[0].ratioString  # e.g. "2/4", "4/4"

        measures = list(part.getElementsByClass('Measure'))

        for i, m in enumerate(measures):
            pattern = get_measure_rhythm(m)
            if pattern:
                rhythm_dict[meter][pattern] += 1

            # If spillover, record 2-measure pattern
            if has_spillover(m) and i + 1 < len(measures):
                next_m = measures[i + 1]
                combined = pattern + get_measure_rhythm(next_m)
                rhythm_dict[meter][combined] += 1


GRID = 1/6

def quantize(d, grid):
    return round(d / grid) * grid

def is_ternary_group(durs, beat=1.0, tol=1e-6):

    if len(durs) != 3:
        return False
    if abs(sum(durs) - beat) > tol:
        return False
    return all(abs(d - beat/3) < tol for d in durs)

def clean_rhythm(pattern, grid=1/6, beat=1.0):
   
    q = [quantize(float(d), grid) for d in pattern]
    q = [d for d in q if d > 0]

    cleaned = []
    i = 0

    while i < len(q):
        # Check for ternary group
        if i + 2 < len(q):
            group = q[i:i+3]
            if is_ternary_group(group, beat):
                cleaned.extend(group)
                i += 3
                continue

    
        d = q[i]

       
        if d < beat / 6 and i + 1 < len(q):
            q[i + 1] += d
        else:
            cleaned.append(d)

        i += 1

    return tuple(cleaned)

from collections import defaultdict

def postprocess_keep_ternary(rhythm_dict, grid=1/6):
    new_dict = defaultdict(lambda: defaultdict(int))

    for meter, patterns in rhythm_dict.items():
        for pattern, count in patterns.items():
            cleaned = clean_rhythm(pattern, grid)
            if cleaned:
                new_dict[meter][cleaned] += count

    return new_dict

processed = postprocess_keep_ternary(rhythm_dict)

new_ocdict = {}
for meter,_ in ocdict.items():
    new_ocdict[meter] = {}
    for key, value in ocdict[meter].items():
        new_key = tuple(x for x in key if x != 0)
        new_ocdict[meter][new_key] = value
        
for p in list(new_ocdict['2/4'].keys()):
    y = list(p)
    for i in y:
        if i == 0:
            print(p)
            continue

for p in list(new_ocdict['12/8'].keys()):
    s = sum(p)
    if s not in [6,12]:
        del new_ocdict['12/8'][p]
for p in list(new_ocdict['9/8'].keys()):
    y = list(p)
    for i in y:
        s = sum(p)
        if s not in [4.5,9]:
            print(p)

for p in list(ocdict['2/4'].keys()):
    y = list(p)
    for i in y:
        if i == 0:
            print(p)
            continue
        
keys = ['4/4','2/4','3/4','3/8','6/8','9/8','5/8','5/4','7/8','7/4','12/8']

for key in list(new_ocdict.keys()):
    if key not in keys:
        del new_ocdict[key]


In [22]:
def get_measure_rhythm(measure):
    rhythm = []
    for el in measure.notesAndRests:
        rhythm.append(el.duration.quarterLength)
    return tuple(rhythm)


In [23]:
def has_spillover(measure):
    for n in measure.notes:
        if n.tie and n.tie.type in ("start", "continue"):
            return True
    return False


In [ ]:

CIPI_PATH = "/Volumes/TOSHIBA  2T/CIPI_dataset/scores"


rhythm_dict = defaultdict(lambda: defaultdict(int))

for root, _, files in os.walk(CIPI_PATH):
    for file in files:
        if file.startswith("._"):
            continue
        
        if not file.lower().endswith((".xml", ".musicxml", ".mid", ".midi")):
            continue

        filepath = os.path.join(root, file)

        try:
            score = converter.parse(filepath)
        except Exception as e:
            print(f"Skipping {file}: {e}")
            continue

        # First staff (right hand)
        part = score.parts[0]

    

        # Get time signatures as a list
        ts = part.recurse().getElementsByClass('TimeSignature')
        if not ts: #if empty list
            continue
        meter = ts[0].ratioString  # e.g. "2/4", "4/4"

        measures = list(part.getElementsByClass('Measure'))

        for i, m in enumerate(measures):
            pattern = get_measure_rhythm(m)
            if pattern:
                rhythm_dict[meter][pattern] += 1

            # If spillover, record 2-measure pattern
            if has_spillover(m) and i + 1 < len(measures):
                next_m = measures[i + 1]
                combined = pattern + get_measure_rhythm(next_m)
                rhythm_dict[meter][combined] += 1




/Users/stepanpshenichnyi/occurence_dict/lib/python3.13/site-packages/music21/musicxml/xmlToM21.py:5491: MusicXMLWarning: Could not import wedge: Error in getting DynamicWedges
  warnings.warn(f'Could not import {tag}: {excep}', MusicXMLWarning)
/Users/stepanpshenichnyi/occurence_dict/lib/python3.13/site-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 77 in part Pianois overfull: 961/480 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/Users/stepanpshenichnyi/occurence_dict/lib/python3.13/site-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 98 in part Pianois overfull: 1441/480 > 3.0,assuming 3.0 is correct.
  warnings.warn(
/Users/stepanpshenichnyi/occurence_dict/lib/python3.13/site-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 278 in part Pianois overfull: 1441/480 > 3.0,assuming 3.0 is correct.
  warnings.warn(
/Users/stepanpshenichnyi/occurence_dict/lib/python3.13/site-packages/music21/musicx

In [ ]:
GRID = 1/6

def quantize(d, grid):
    return round(d / grid) * grid

def is_ternary_group(durs, beat=1.0, tol=1e-6):

    if len(durs) != 3:
        return False
    if abs(sum(durs) - beat) > tol:
        return False
    return all(abs(d - beat/3) < tol for d in durs)



In [ ]:
def clean_rhythm(pattern, grid=1/6, beat=1.0):
   
    q = [quantize(float(d), grid) for d in pattern]
    q = [d for d in q if d > 0]

    cleaned = []
    i = 0

    while i < len(q):
        # Check for ternary group
        if i + 2 < len(q):
            group = q[i:i+3]
            if is_ternary_group(group, beat):
                cleaned.extend(group)
                i += 3
                continue

    
        d = q[i]

       
        if d < beat / 6 and i + 1 < len(q):
            q[i + 1] += d
        else:
            cleaned.append(d)

        i += 1

    return tuple(cleaned)


In [17]:
from collections import defaultdict

def postprocess_keep_ternary(rhythm_dict, grid=1/6):
    new_dict = defaultdict(lambda: defaultdict(int))

    for meter, patterns in rhythm_dict.items():
        for pattern, count in patterns.items():
            cleaned = clean_rhythm(pattern, grid)
            if cleaned:
                new_dict[meter][cleaned] += count

    return new_dict


In [18]:
processed = postprocess_keep_ternary(rhythm_dict)


In [26]:
rhythm_dict

defaultdict(<function __main__.<lambda>()>,
            {'2/4': defaultdict(int,
                         {(2.0,): 618,
                          (1.0, 0.5, 0.125, 0.125, 0.125, 0.125): 24,
                          (0.5, 0.5, 0.5, 0.5): 1770,
                          (0.5, 1.0, 0.125, 0.125, 0.125, 0.125): 6,
                          (0.5, 1.5): 27,
                          (0.125,
                           0.125,
                           0.125,
                           0.125,
                           0.5,
                           0.125,
                           0.125,
                           0.125,
                           0.125,
                           0.5): 48,
                          (0.5, 0.5, 0.125, 0.125, 0.125, 0.125, 0.5): 16,
                          (0.5, 0.5, 0.5, 0.5, 2.0): 4,
                          (0.5,
                           Fraction(1, 6),
                           Fraction(1, 6),
                           Fraction(1, 6),
            

In [ ]:
for d in rhythm_dict['d']:
    d = dict(d)


In [83]:


with open("rhythm_dict_with_cleanup_20dec25.pkl", "wb") as f:
    pickle.dump(dict(new_ocdict), f)


In [8]:
with open('rhythm_dict_without_cleanup.pkl', 'rb') as f:
    ocdict = pickle.load(f)

In [9]:
ocdict

{'2/4': {(2.0,): 618,
  (1.0, 0.5, 0.125, 0.125, 0.125, 0.125): 24,
  (0.5, 0.5, 0.5, 0.5): 1770,
  (0.5, 1.0, 0.125, 0.125, 0.125, 0.125): 6,
  (0.5, 1.5): 27,
  (0.125, 0.125, 0.125, 0.125, 0.5, 0.125, 0.125, 0.125, 0.125, 0.5): 48,
  (0.5, 0.5, 0.125, 0.125, 0.125, 0.125, 0.5): 16,
  (0.5, 0.5, 0.5, 0.5, 2.0): 4,
  (0.5, Fraction(1, 6), Fraction(1, 6), Fraction(1, 6), 1.0): 8,
  (0.5, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25): 84,
  (0.25, 0.25, 0.25, 0.25, 0.25, 0.75): 9,
  (0.25, 0.75, 1.0): 9,
  (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25): 1953,
  (0.25, 0.125, 0.0625, 0.0625, 1.0, 0.5): 8,
  (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.375, 0.125): 5,
  (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.5): 33,
  (0.25, 0.25, 0.5, 0.5, 0.375, 0.0625, 0.0625): 2,
  (0.5,): 41,
  (1.5, 0.25, 0.25): 38,
  (0.5, 0.5, 0.25, 0.25, 0.25, 0.25): 173,
  (1.0, 0.5, 0.25, 0.25): 51,
  (0.5, 0.5, 0.5, 0.25, 0.25): 48,
  (0.25, 0.25, 0.25, 0.25, 0.5, 0.25, 0.25): 25,
  (1.0, 0.5): 48,
  (1.0, 0.5, 0.5): 464,
 

In [4]:
list(ocdict.keys())

['2/4',
 '3/4',
 '6/8',
 '8/8',
 '4/4',
 '2/2',
 '12/8',
 '2/8',
 '3/8',
 '6/16',
 '12/16',
 '6/4',
 '9/8',
 '4/2',
 '4/8',
 '3/2',
 '9/16',
 '10/8',
 '24/16',
 '5/4']

In [45]:
d = {
    (0.5, 0.25, 0.25, 0.00, 0.25, 0.0, 0.25): 84,
    (0.25, 0.25, 0.25, 0.25, 0.25, 0.75): 9,
    (0.25, 0.0, 1.0): 9,
    (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25): 1953,
    (0.25, 0.125, 0.0, 0.0, 1.0, 0.5): 8,
    (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.0, 0.125): 5,
}

# Create a new dictionary to avoid modifying while iterating
new_d = {}
for key, value in d.items():
    new_key = tuple(x for x in key if x != 0)
    new_d[new_key] = value

d = new_d
print(d)

{(0.5, 0.25, 0.25, 0.25, 0.25): 84, (0.25, 0.25, 0.25, 0.25, 0.25, 0.75): 9, (0.25, 1.0): 9, (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25): 1953, (0.25, 0.125, 1.0, 0.5): 8, (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.125): 5}


In [49]:
new_ocdict = {}
for meter,_ in ocdict.items():
    new_ocdict[meter] = {}
    for key, value in ocdict[meter].items():
        new_key = tuple(x for x in key if x != 0)
        new_ocdict[meter][new_key] = value

new_ocdict['2/4']



{(2.0,): 13,
 (1.0, 0.5, 0.125, 0.125, 0.125, 0.125): 24,
 (0.5, 0.5, 0.5, 0.5): 1,
 (0.5, 1.0, 0.125, 0.125, 0.125, 0.125): 6,
 (0.5, 1.5): 27,
 (0.125, 0.125, 0.125, 0.125, 0.5, 0.125, 0.125, 0.125, 0.125, 0.5): 48,
 (0.5, 0.5, 0.125, 0.125, 0.125, 0.125, 0.5): 16,
 (0.5, 0.5, 0.5, 0.5, 2.0): 4,
 (0.5, Fraction(1, 6), Fraction(1, 6), Fraction(1, 6), 1.0): 8,
 (0.5, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25): 20,
 (0.25, 0.25, 0.25, 0.25, 0.25, 0.75): 9,
 (0.25, 0.75, 1.0): 9,
 (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25): 4,
 (0.25, 0.125, 0.0625, 0.0625, 1.0, 0.5): 8,
 (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.375, 0.125): 5,
 (0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.5): 2,
 (0.25, 0.25, 0.5, 0.5, 0.375, 0.0625, 0.0625): 2,
 (0.5,): 41,
 (1.5, 0.25, 0.25): 4,
 (0.5, 0.5, 0.25, 0.25, 0.25, 0.25): 6,
 (1.0, 0.5, 0.25, 0.25): 1,
 (0.5, 0.5, 0.5, 0.25, 0.25): 1,
 (0.25, 0.25, 0.25, 0.25, 0.5, 0.25, 0.25): 25,
 (1.0, 0.5): 48,
 (1.0, 0.5, 0.5): 1,
 (0.75, 0.25, 0.75, 0.25, 0.75, 0.25): 14,
 (0.7

In [ ]:
new_ocdict = {}
for meter,_ in ocdict.items():
    new_ocdict[meter] = {}
    for key, value in ocdict[meter].items():
        new_key = tuple(x for x in key if x != 0)
        new_ocdict[meter][new_key] = value
        
for p in list(new_ocdict['2/4'].keys()):
    y = list(p)
    for i in y:
        if i == 0:
            print(p)
            continue
   

            

In [ ]:
new_ocdict = {}
for meter,_ in ocdict.items():
    new_ocdict[meter] = {}
    for key, value in ocdict[meter].items():
        new_key = tuple(x for x in key if x != 0)
        new_ocdict[meter][new_key] = value
        
for p in list(new_ocdict['2/4'].keys()):
    y = list(p)
    for i in y:
        if i == 0:
            print(p)
            continue

for p in list(new_ocdict['12/8'].keys()):
    s = sum(p)
    if s not in [6,12]:
        del new_ocdict['12/8'][p]
for p in list(new_ocdict['9/8'].keys()):
    y = list(p)
    for i in y:
        s = sum(p)
        if s not in [4.5,9]:
            print(p)

for p in list(ocdict['2/4'].keys()):
    y = list(p)
    for i in y:
        if i == 0:
            print(p)
            continue
        
keys = ['4/4','2/4','3/4','3/8','6/8','9/8','5/8','5/4','7/8','7/4','12/8']

for key in list(new_ocdict.keys()):
    if key not in keys:
        del new_ocdict[key]

In [82]:
for p in list(new_ocdict['12/8'].keys()):
    s = sum(p)
    if s not in [6,12]:
        del new_ocdict['12/8'][p]

In [79]:
for p in list(new_ocdict['9/8'].keys()):
    y = list(p)
    for i in y:
        s = sum(p)
        if s not in [4.5,9]:
            print(p)
            

In [ ]:
for p in list(ocdict['2/4'].keys()):
    y = list(p)
    for i in y:
        if i == 0:
            print(p)
            continue

In [61]:
ocdict.keys()

dict_keys(['2/4', '3/4', '6/8', '8/8', '4/4', '2/2', '12/8', '2/8', '3/8', '6/16', '12/16', '6/4', '9/8', '4/2', '4/8', '3/2', '9/16', '10/8', '24/16', '5/4'])

In [58]:
for key,_ in ocdict.items():
    if key not in keys:
        print(key)

8/8
2/2
2/8
6/16
12/16
6/4
4/2
4/8
3/2
9/16
10/8
24/16


In [57]:
keys = ['4/4','2/4','3/4','3/8','6/8','9/8','5/8','5/4','7/8','7/4','12/8']

In [60]:
keys = ['4/4','2/4','3/4','3/8','6/8','9/8','5/8','5/4','7/8','7/4','12/8']

for key in list(new_ocdict.keys()):
    if key not in keys:
        del new_ocdict[key]

new_ocdict.keys()

dict_keys(['2/4', '3/4', '6/8', '4/4', '12/8', '3/8', '9/8', '5/4'])

In [62]:
list(ocdict.keys())

['2/4',
 '3/4',
 '6/8',
 '8/8',
 '4/4',
 '2/2',
 '12/8',
 '2/8',
 '3/8',
 '6/16',
 '12/16',
 '6/4',
 '9/8',
 '4/2',
 '4/8',
 '3/2',
 '9/16',
 '10/8',
 '24/16',
 '5/4']

20 dec: removed 0.0 values, removed meters not needed, removed patterns that don't add up